# DL 모델 비교 실험 — 10분 전력 데이터 기반 단기 예측

**데이터**: ~/data/KIER_3_Temporal_Resolution/seasonal_naive/KIER_USAGE_ELEC_INST_INTERP_10MIN.csv`  
**모델**: 1D-CNN LSTM · GRU · Transformer · TCN · RetNet  
**예측**: 단일 세대별 10분 후 전력 소비량 (1-step ahead)  
**평가 지표**: MAE · RMSE · MAPE · R²  
**결과 저장**: `results/models/`

---

| 항목 | 설정 |
|---|---|
| Lookback 윈도우 | 144 스텝 (24H) |
| 예측 horizon | 1 스텝 (10분 후) |
| Train / Val / Test | 70 / 15 / 15 % (시간순 분할) |
| 세대 샘플 수 | 10개 (CFG에서 조정) |
| 정규화 | 세대별 Min-Max |
| 시간 특징 | hour, day-of-week, month (sin/cos 인코딩) |

In [ ]:
# ── 00. Configuration ────────────────────────────────────────────────────────
from pathlib import Path

CFG = {
    # 데이터
    'data_path'     : str(Path.home() / 'data' / 'KIER_3_Temporal_Resolution' / 'seasonal_naive' / 'KIER_USAGE_ELEC_INST_INTERP_10MIN.csv'),
    'n_households'  : 10,          # 실험에 사용할 세대 수 (전체 348세대 중 랜덤 선택)
    'seed'          : 42,
    # 시퀀스
    'seq_len'       : 144,         # 룩백: 144 × 10분 = 24시간
    'pred_horizon'  : 1,           # 1-step ahead (10분 후)
    # 분할 비율
    'train_ratio'   : 0.70,
    'val_ratio'     : 0.15,        # test_ratio = 1 - train - val = 0.15
    # 학습
    'epochs'        : 200,
    'batch_size'    : 128,
    'patience'      : 20,          # EarlyStopping
    'lr'            : 1e-3,
    # 출력
    'results_dir'   : '../results/models/',
}

In [ ]:
# ── 01. Imports ───────────────────────────────────────────────────────────────
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

import tensorflow as tf
from tensorflow import keras
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
np.random.seed(CFG['seed'])
tf.random.set_seed(CFG['seed'])

# core 경로 추가
sys.path.insert(0, os.path.abspath('..'))

# GPU 확인
gpus = tf.config.list_physical_devices('GPU')
print(f"TF {tf.__version__}  |  GPU: {[g.name for g in gpus] if gpus else 'CPU only'}")

# 결과 디렉토리
RESULTS_DIR = Path(CFG['results_dir'])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 02. 데이터 로드 & 특징 공학

In [ ]:
# ── 데이터 로드 ───────────────────────────────────────────────────────────────
raw = pd.read_csv(CFG['data_path'], parse_dates=['METER_DATE'], index_col='METER_DATE')
raw = raw.sort_index()

print(f"기간 : {raw.index[0]}  →  {raw.index[-1]}")
print(f"행수 : {len(raw):,}   세대 수: {raw.shape[1]}")
raw.head(3)

In [ ]:
# ── 시간 특징 생성 (cyclical encoding) ───────────────────────────────────────
def make_time_features(index: pd.DatetimeIndex) -> pd.DataFrame:
    """hour / day-of-week / month → sin·cos 인코딩 (6개 특징)"""
    t = pd.DataFrame(index=index)
    t['hour_sin']  = np.sin(2 * np.pi * index.hour / 24)
    t['hour_cos']  = np.cos(2 * np.pi * index.hour / 24)
    t['dow_sin']   = np.sin(2 * np.pi * index.dayofweek / 7)
    t['dow_cos']   = np.cos(2 * np.pi * index.dayofweek / 7)
    t['month_sin'] = np.sin(2 * np.pi * index.month / 12)
    t['month_cos'] = np.cos(2 * np.pi * index.month / 12)
    return t

TIME_FEATS = make_time_features(raw.index)   # (T, 6)
N_TIME_FEATS = TIME_FEATS.shape[1]           # 6
print(f"시간 특징: {list(TIME_FEATS.columns)}")

In [ ]:
# ── 실험 세대 선택 ────────────────────────────────────────────────────────────
# NaN이 없는 세대만 후보로 사용
complete_cols = [c for c in raw.columns if raw[c].isna().sum() == 0]
rng = np.random.default_rng(CFG['seed'])
selected_cols = list(rng.choice(complete_cols, size=CFG['n_households'], replace=False))
print(f"선택된 세대 ({CFG['n_households']}개): {selected_cols}")

## 03. 데이터셋 구성 유틸리티

In [ ]:
def build_dataset_for_household(series: pd.Series, time_feats: pd.DataFrame,
                                seq_len: int, train_r: float, val_r: float):
    """
    단일 세대 시계열 → (X_train, y_train, X_val, y_val, X_test, y_test, scaler)

    특징 = [에너지(정규화) + 시간 특징 6개]  →  n_features = 7
    타겟 = 에너지(정규화)  →  단일 값
    """
    T = len(series)
    n_train = int(T * train_r)
    n_val   = int(T * val_r)

    # Min-Max 정규화 (훈련 구간 기준)
    scaler = MinMaxScaler()
    energy = series.values.reshape(-1, 1)
    scaler.fit(energy[:n_train])
    energy_scaled = scaler.transform(energy)   # (T, 1)

    # 특징 행렬: [energy_scaled | time_feats]
    feat = np.concatenate([energy_scaled, time_feats.values], axis=1)  # (T, 7)
    target = energy_scaled.ravel()                                       # (T,)

    # 슬라이딩 윈도우
    def make_windows(feat_arr, target_arr, start, end):
        X, y = [], []
        for i in range(start, end - seq_len):
            X.append(feat_arr[i : i + seq_len])
            y.append(target_arr[i + seq_len])
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

    X_tr, y_tr = make_windows(feat, target, 0,            n_train)
    X_va, y_va = make_windows(feat, target, n_train,      n_train + n_val)
    X_te, y_te = make_windows(feat, target, n_train + n_val, T)

    return X_tr, y_tr, X_va, y_va, X_te, y_te, scaler


# 특징 수 확인
_test_s = raw[selected_cols[0]]
_res = build_dataset_for_household(_test_s, TIME_FEATS, CFG['seq_len'],
                                    CFG['train_ratio'], CFG['val_ratio'])
X_tr_ex, y_tr_ex = _res[0], _res[1]
N_FEATURES = X_tr_ex.shape[2]
SEQ_LEN    = X_tr_ex.shape[1]
print(f"X_train shape : {X_tr_ex.shape}   (samples, seq_len={SEQ_LEN}, n_features={N_FEATURES})")
print(f"y_train shape : {y_tr_ex.shape}")

## 04. 모델 빌더 (5종)

| 모델 | 핵심 특징 |
|---|---|
| **1D-CNN LSTM** | CNN으로 로컬 패턴 추출 → LSTM 장기 의존성 |
| **GRU** | LSTM 대비 경량, 빠른 수렴 |
| **Transformer** | Self-Attention + Sinusoidal positional encoding |
| **TCN** | 팽창 인과 합성곱, 수용 영역 = 2×(k-1)×Σdilations |
| **RetNet** | Multi-Scale Retention, Transformer 대비 O(T) 추론 |

In [ ]:
# ── 4-1. 1D CNN-LSTM ──────────────────────────────────────────────────────────
def build_1dcnn_lstm(n_features: int, seq_len: int, activation: str = 'swish'):
    inp = keras.Input(shape=(seq_len, n_features))

    x = keras.layers.Conv1D(64, 3, padding='causal', activation=activation)(inp)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Conv1D(128, 3, padding='causal', activation=activation)(x)
    x = keras.layers.BatchNormalization()(x)

    x = keras.layers.LSTM(128, return_sequences=True, dropout=0.15)(x)
    x = keras.layers.LSTM(64,  return_sequences=False, dropout=0.15)(x)

    x = keras.layers.Dense(32, activation=activation)(x)
    out = keras.layers.Dense(1)(x)
    return keras.Model(inp, out, name='1D_CNN_LSTM')


# ── 4-2. GRU ──────────────────────────────────────────────────────────────────
def build_gru(n_features: int, seq_len: int):
    inp = keras.Input(shape=(seq_len, n_features))
    x = keras.layers.GRU(128, return_sequences=True, dropout=0.15)(inp)
    x = keras.layers.LayerNormalization()(x)
    x = keras.layers.GRU(64,  return_sequences=False, dropout=0.15)(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    out = keras.layers.Dense(1)(x)
    return keras.Model(inp, out, name='GRU')


# ── 4-3. Transformer ──────────────────────────────────────────────────────────
def _pos_encoding(seq_len: int, d_model: int) -> tf.Tensor:
    positions = np.arange(seq_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angles = positions / np.power(10000.0, (2 * (dims // 2)) / d_model)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis], dtype=tf.float32)   # (1, seq_len, d_model)


def build_transformer(n_features: int, seq_len: int,
                      d_model: int = 64, num_heads: int = 4,
                      ff_dim: int = 128, num_layers: int = 2,
                      dropout: float = 0.1):
    inp = keras.Input(shape=(seq_len, n_features))
    x   = keras.layers.Dense(d_model)(inp)
    x   = x + _pos_encoding(seq_len, d_model)
    x   = keras.layers.Dropout(dropout)(x)

    for _ in range(num_layers):
        attn = keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout)(x, x)
        x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn)
        ffn = keras.layers.Dense(ff_dim, activation='relu')(x)
        ffn = keras.layers.Dense(d_model)(ffn)
        ffn = keras.layers.Dropout(dropout)(ffn)
        x   = keras.layers.LayerNormalization(epsilon=1e-6)(x + ffn)

    x   = keras.layers.GlobalAveragePooling1D()(x)
    x   = keras.layers.Dense(32, activation='relu')(x)
    out = keras.layers.Dense(1)(x)
    return keras.Model(inp, out, name='Transformer')


# ── 4-4. TCN ──────────────────────────────────────────────────────────────────
def _tcn_block(x, filters, kernel_size, dilation, dropout, activation):
    residual = x
    for _ in range(2):
        x = keras.layers.Conv1D(filters, kernel_size, padding='causal',
                                dilation_rate=dilation)(x)
        x = keras.layers.LayerNormalization()(x)
        x = keras.layers.Activation(activation)(x)
        x = keras.layers.Dropout(dropout)(x)
    if residual.shape[-1] != filters:
        residual = keras.layers.Conv1D(filters, 1)(residual)
    return keras.layers.Add()([x, residual])


def build_tcn(n_features: int, seq_len: int,
              nb_filters: int = 64, kernel_size: int = 3,
              dilations: list | None = None, dropout: float = 0.1,
              activation: str = 'relu'):
    if dilations is None:
        dilations = [1, 2, 4, 8, 16]
    inp = keras.Input(shape=(seq_len, n_features))
    x   = inp
    for d in dilations:
        x = _tcn_block(x, nb_filters, kernel_size, d, dropout, activation)
    x   = keras.layers.GlobalAveragePooling1D()(x)
    x   = keras.layers.Dense(32, activation=activation)(x)
    out = keras.layers.Dense(1)(x)
    return keras.Model(inp, out, name='TCN')


# ── 4-5. RetNet ───────────────────────────────────────────────────────────────
class _MultiScaleRetention(keras.layers.Layer):
    """Parallel-mode Multi-Scale Retention (Sun et al. 2023 §3)."""

    def __init__(self, d_model: int, num_heads: int, **kwargs):
        super().__init__(**kwargs)
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = d_model // num_heads
        self.d_model   = d_model
        self.gammas    = [1.0 - 2.0 ** (-5 - h) for h in range(num_heads)]

        self.W_Q = keras.layers.Dense(d_model, use_bias=False)
        self.W_K = keras.layers.Dense(d_model, use_bias=False)
        self.W_V = keras.layers.Dense(d_model, use_bias=False)
        self.W_G = keras.layers.Dense(d_model)
        self.W_O = keras.layers.Dense(d_model)
        self.ln  = keras.layers.LayerNormalization()

    def call(self, x, training=None):
        B, T = tf.shape(x)[0], tf.shape(x)[1]
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        G = tf.nn.swish(self.W_G(x))

        def split_heads(z):
            z = tf.reshape(z, [B, T, self.num_heads, self.head_dim])
            return tf.transpose(z, [0, 2, 1, 3])

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)
        idx  = tf.cast(tf.range(T), tf.float32)
        diff = idx[:, tf.newaxis] - idx[tf.newaxis, :]
        causal = tf.cast(diff >= 0, tf.float32)
        scale  = tf.cast(self.head_dim, tf.float32) ** 0.5

        heads_out = []
        for h in range(self.num_heads):
            D      = tf.pow(self.gammas[h], tf.maximum(diff, 0.0)) * causal
            scores = tf.matmul(Q[:, h], K[:, h], transpose_b=True) / scale
            ret_h  = tf.matmul(scores * D[tf.newaxis], V[:, h])
            heads_out.append(ret_h)

        ret = tf.stack(heads_out, axis=2)
        ret = tf.reshape(ret, [B, T, self.d_model])
        return self.W_O(self.ln(ret) * G)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'d_model': self.d_model, 'num_heads': self.num_heads})
        return cfg


def build_retnet(n_features: int, seq_len: int,
                 d_model: int = 64, num_heads: int = 4,
                 ff_dim: int = 128, num_layers: int = 2,
                 dropout: float = 0.1):
    inp = keras.Input(shape=(seq_len, n_features))
    x   = keras.layers.Dense(d_model)(inp)
    x   = keras.layers.Dropout(dropout)(x)

    for _ in range(num_layers):
        residual = x
        ret = _MultiScaleRetention(d_model, num_heads)(x)
        ret = keras.layers.Dropout(dropout)(ret)
        x   = keras.layers.LayerNormalization()(residual + ret)

        residual = x
        ffn = keras.layers.Dense(ff_dim, activation='gelu')(x)
        ffn = keras.layers.Dense(d_model)(ffn)
        ffn = keras.layers.Dropout(dropout)(ffn)
        x   = keras.layers.LayerNormalization()(residual + ffn)

    x   = keras.layers.GlobalAveragePooling1D()(x)
    x   = keras.layers.Dense(32, activation='gelu')(x)
    out = keras.layers.Dense(1)(x)
    return keras.Model(inp, out, name='RetNet')


# ── 모델 레지스트리 ───────────────────────────────────────────────────────────
MODEL_REGISTRY = {
    '1D_CNN_LSTM' : lambda nf, sl: build_1dcnn_lstm(nf, sl),
    'GRU'         : lambda nf, sl: build_gru(nf, sl),
    'Transformer' : lambda nf, sl: build_transformer(nf, sl),
    'TCN'         : lambda nf, sl: build_tcn(nf, sl),
    'RetNet'      : lambda nf, sl: build_retnet(nf, sl),
}

# 파라미터 수 확인
print("모델별 파라미터 수:")
for name, builder in MODEL_REGISTRY.items():
    m = builder(N_FEATURES, SEQ_LEN)
    print(f"  {name:<15} {m.count_params():>10,}")
    keras.backend.clear_session()

## 05. 학습 루프 — 세대별 × 모델별

In [ ]:
def mape(y_true, y_pred, eps=1e-8):
    return np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100


def compute_metrics(y_true, y_pred, scaler):
    """정규화 역변환 후 실제 단위(kWh)로 지표 계산."""
    yt = scaler.inverse_transform(y_true.reshape(-1, 1)).ravel()
    yp = scaler.inverse_transform(y_pred.reshape(-1, 1)).ravel()
    return {
        'MAE'  : mean_absolute_error(yt, yp),
        'RMSE' : np.sqrt(mean_squared_error(yt, yp)),
        'MAPE' : mape(yt, yp),
        'R2'   : r2_score(yt, yp),
    }


def train_model(model, X_tr, y_tr, X_va, y_va, cfg):
    """모델 컴파일·학습, 학습 히스토리 반환."""
    model.compile(
        loss=keras.losses.Huber(),
        optimizer=keras.optimizers.Adam(learning_rate=cfg['lr'], clipnorm=1.0),
        metrics=['mae'],
    )
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=cfg['patience'],
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10,
                          min_lr=1e-6, verbose=0),
    ]
    t0 = time.time()
    hist = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=cfg['epochs'], batch_size=cfg['batch_size'],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0
    return hist, elapsed

In [ ]:
# ── 본 실험: 세대 × 모델 루프 ─────────────────────────────────────────────────
records   = []    # [{model, household, MAE, RMSE, MAPE, R2, elapsed_s, n_epochs}, ...]
histories = {}    # {model_name: [hist1, hist2, ...]}
sample_preds = {} # {model_name: (y_true, y_pred)} - 마지막 세대 예측 저장 (시각화용)

for col in tqdm(selected_cols, desc='Household'):
    series = raw[col]
    X_tr, y_tr, X_va, y_va, X_te, y_te, scaler = build_dataset_for_household(
        series, TIME_FEATS, CFG['seq_len'], CFG['train_ratio'], CFG['val_ratio'])

    for model_name, builder in MODEL_REGISTRY.items():
        model = builder(N_FEATURES, SEQ_LEN)
        hist, elapsed = train_model(model, X_tr, y_tr, X_va, y_va, CFG)

        y_pred = model.predict(X_te, verbose=0).ravel()
        metrics = compute_metrics(y_te, y_pred, scaler)

        records.append({
            'model'     : model_name,
            'household' : col,
            'n_epochs'  : len(hist.history['loss']),
            'elapsed_s' : round(elapsed, 1),
            **metrics,
        })

        if model_name not in histories:
            histories[model_name] = []
        histories[model_name].append(hist.history)

        # 마지막 세대 예측 저장
        if col == selected_cols[-1]:
            sample_preds[model_name] = (
                scaler.inverse_transform(y_te.reshape(-1,1)).ravel(),
                scaler.inverse_transform(y_pred.reshape(-1,1)).ravel(),
            )

        keras.backend.clear_session()

results_df = pd.DataFrame(records)
print("\n실험 완료  |  총 레코드:", len(results_df))

## 06. 성능 비교 집계

In [ ]:
# ── 모델별 평균·표준편차 ──────────────────────────────────────────────────────
METRICS = ['MAE', 'RMSE', 'MAPE', 'R2']

summary = results_df.groupby('model')[METRICS + ['elapsed_s', 'n_epochs']].agg(['mean', 'std'])
summary.columns = ['_'.join(c) for c in summary.columns]

# 보기 좋은 요약 테이블
display_rows = []
for model_name in MODEL_REGISTRY.keys():
    row = summary.loc[model_name]
    display_rows.append({
        'Model'    : model_name,
        'MAE'      : f"{row['MAE_mean']:.4f} ± {row['MAE_std']:.4f}",
        'RMSE'     : f"{row['RMSE_mean']:.4f} ± {row['RMSE_std']:.4f}",
        'MAPE (%)'  : f"{row['MAPE_mean']:.2f} ± {row['MAPE_std']:.2f}",
        'R²'       : f"{row['R2_mean']:.4f} ± {row['R2_std']:.4f}",
        'Time (s)' : f"{row['elapsed_s_mean']:.1f}",
        'Epochs'   : f"{row['n_epochs_mean']:.0f}",
    })

display_df = pd.DataFrame(display_rows).set_index('Model')
print("\n=== DL 모델 비교 결과 (평균 ± 표준편차, 세대 수:", CFG['n_households'], ") ===")
display(display_df)

In [ ]:
# ── 순위 산정 (낮을수록 좋음: MAE, RMSE, MAPE / 높을수록 좋음: R²) ────────────
rank_df = pd.DataFrame(index=list(MODEL_REGISTRY.keys()))
for m in MODEL_REGISTRY:
    r = summary.loc[m]
    rank_df.loc[m, 'MAE_mean']  = r['MAE_mean']
    rank_df.loc[m, 'RMSE_mean'] = r['RMSE_mean']
    rank_df.loc[m, 'MAPE_mean'] = r['MAPE_mean']
    rank_df.loc[m, 'R2_mean']   = r['R2_mean']

rank_df['rank_MAE']  = rank_df['MAE_mean'].rank()
rank_df['rank_RMSE'] = rank_df['RMSE_mean'].rank()
rank_df['rank_MAPE'] = rank_df['MAPE_mean'].rank()
rank_df['rank_R2']   = rank_df['R2_mean'].rank(ascending=False)   # R² 높을수록 좋음
rank_df['avg_rank']  = rank_df[['rank_MAE','rank_RMSE','rank_MAPE','rank_R2']].mean(axis=1)

print("\n=== 종합 순위 (avg_rank 낮을수록 우수) ===")
display(rank_df.sort_values('avg_rank')[['MAE_mean','RMSE_mean','MAPE_mean','R2_mean','avg_rank']]
        .rename(columns={'MAE_mean':'MAE','RMSE_mean':'RMSE','MAPE_mean':'MAPE(%)','R2_mean':'R²','avg_rank':'Avg Rank'}))

## 07. 시각화

In [ ]:
# ── 7-1. 지표별 Bar Chart ────────────────────────────────────────────────────
model_order = list(MODEL_REGISTRY.keys())
palette = sns.color_palette('tab10', n_colors=len(model_order))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('DL 모델 성능 비교 (평균 ± 표준편차)', fontsize=14, fontweight='bold')

for ax, metric in zip(axes.ravel(), METRICS):
    means = [summary.loc[m, f'{metric}_mean'] for m in model_order]
    stds  = [summary.loc[m, f'{metric}_std']  for m in model_order]
    bars  = ax.bar(model_order, means, yerr=stds, capsize=5,
                   color=palette, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel('kWh' if metric != 'MAPE' else '%')
    ax.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'dl_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7-2. 학습 곡선 (val_loss, 세대 평균) ─────────────────────────────────────
fig, axes = plt.subplots(1, len(MODEL_REGISTRY), figsize=(18, 4), sharey=True)
fig.suptitle('학습 곡선 (Validation Loss, 세대 평균)', fontsize=13, fontweight='bold')

for ax, (model_name, hists) in zip(axes, histories.items()):
    max_ep = max(len(h['val_loss']) for h in hists)
    mat = np.full((len(hists), max_ep), np.nan)
    for i, h in enumerate(hists):
        n = len(h['val_loss'])
        mat[i, :n] = h['val_loss']

    mean_loss = np.nanmean(mat, axis=0)
    std_loss  = np.nanstd(mat, axis=0)
    epochs    = np.arange(1, max_ep + 1)

    ax.plot(epochs, mean_loss, linewidth=1.5, label='mean val_loss')
    ax.fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss, alpha=0.25)
    ax.set_title(model_name, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Huber Loss')

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'dl_comparison_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7-3. 예측 곡선 비교 (마지막 세대, 최초 72H = 432 스텝) ────────────────────
PLOT_STEPS = 432   # 72시간 미리보기

fig, axes = plt.subplots(len(MODEL_REGISTRY), 1, figsize=(16, 3 * len(MODEL_REGISTRY)),
                         sharex=True)
fig.suptitle(f'예측 vs 실제 (세대: {selected_cols[-1]}, 최초 {PLOT_STEPS//6}H)',
             fontsize=13, fontweight='bold')

for ax, (model_name, (y_true, y_pred)) in zip(axes, sample_preds.items()):
    n = min(PLOT_STEPS, len(y_true))
    ax.plot(y_true[:n], label='Actual',    color='steelblue', linewidth=0.8, alpha=0.9)
    ax.plot(y_pred[:n], label='Predicted', color='tomato',    linewidth=0.8, alpha=0.85,
            linestyle='--')
    ax.set_title(model_name, fontweight='bold')
    ax.set_ylabel('kWh')
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Step (10분 단위)')
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'dl_comparison_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7-4. Radar Chart (정규화된 지표 종합 비교) ────────────────────────────────
from matplotlib.patches import FancyArrowPatch

radar_metrics  = ['MAE', 'RMSE', 'MAPE', 'R2']
N_met = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N_met, endpoint=False).tolist()
angles += angles[:1]

# 정규화 (0=최악, 1=최우수)
radar_vals = {}
for m_name in model_order:
    raw_v = [summary.loc[m_name, f'{met}_mean'] for met in radar_metrics]
    radar_vals[m_name] = raw_v

radar_arr = np.array([radar_vals[m] for m in model_order])   # (n_models, n_metrics)
mins = radar_arr.min(axis=0)
maxs = radar_arr.max(axis=0)

def normalize_metric(v, lo, hi, higher_is_better=False):
    if hi == lo:
        return 0.5
    norm = (v - lo) / (hi - lo)   # 0=worst, 1=best (for lower-is-better)
    return norm if higher_is_better else (1 - norm)

higher_is_better_flags = [False, False, False, True]  # MAE, RMSE, MAPE, R²

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_title('모델 종합 성능 (Radar Chart)', fontsize=13, fontweight='bold', pad=20)

for idx, m_name in enumerate(model_order):
    norm_v = [
        normalize_metric(radar_vals[m_name][j], mins[j], maxs[j], higher_is_better_flags[j])
        for j in range(N_met)
    ]
    norm_v += norm_v[:1]
    ax.plot(angles, norm_v, linewidth=2, label=m_name, color=palette[idx])
    ax.fill(angles, norm_v, alpha=0.08, color=palette[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%','50%','75%','100%'], fontsize=7, color='grey')
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)

fig.savefig(RESULTS_DIR / 'dl_comparison_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 08. 결과 저장

In [ ]:
# ── 세부 결과 CSV ─────────────────────────────────────────────────────────────
results_df.to_csv(RESULTS_DIR / 'dl_comparison_detail.csv', index=False)

# ── 모델별 집계 결과 CSV ──────────────────────────────────────────────────────
agg_out = []
for m_name in model_order:
    r = summary.loc[m_name]
    agg_out.append({
        'model'         : m_name,
        'MAE_mean'      : round(r['MAE_mean'],  4),
        'MAE_std'       : round(r['MAE_std'],   4),
        'RMSE_mean'     : round(r['RMSE_mean'], 4),
        'RMSE_std'      : round(r['RMSE_std'],  4),
        'MAPE_mean'     : round(r['MAPE_mean'], 2),
        'MAPE_std'      : round(r['MAPE_std'],  2),
        'R2_mean'       : round(r['R2_mean'],   4),
        'R2_std'        : round(r['R2_std'],    4),
        'elapsed_s_mean': round(r['elapsed_s_mean'], 1),
        'n_epochs_mean' : round(r['n_epochs_mean'],  1),
    })

agg_df = pd.DataFrame(agg_out)
agg_df.to_csv(RESULTS_DIR / 'dl_comparison_summary.csv', index=False)

# ── 순위 CSV ──────────────────────────────────────────────────────────────────
rank_df.sort_values('avg_rank').to_csv(RESULTS_DIR / 'dl_comparison_rank.csv')

print("저장 완료:")
for p in sorted(RESULTS_DIR.glob('dl_comparison*')):
    print(f"  {p.relative_to(Path('../'))}")

## 09. 실험 요약

아래 셀을 실행하면 최종 권장 모델과 근거가 출력됩니다.

In [ ]:
best_model = rank_df['avg_rank'].idxmin()
best_row   = summary.loc[best_model]

print("="*55)
print(f"  최우수 모델  :  {best_model}")
print(f"  MAE          :  {best_row['MAE_mean']:.4f} kWh")
print(f"  RMSE         :  {best_row['RMSE_mean']:.4f} kWh")
print(f"  MAPE         :  {best_row['MAPE_mean']:.2f} %")
print(f"  R²           :  {best_row['R2_mean']:.4f}")
print(f"  평균 학습시간  :  {best_row['elapsed_s_mean']:.0f} s")
print("="*55)
print()
print("세부 결과 → results/models/dl_comparison_detail.csv")
print("집계 결과 → results/models/dl_comparison_summary.csv")
print("순위      → results/models/dl_comparison_rank.csv")